
# 03. Modelos baseline — dataset enriquecido de contratos

Usa exclusivamente:

- `dataset_modelagem_obras_snapshots_enriquecido.csv`
- `dicionario_dataset_modelagem_enriquecido.csv`

Regras:
- regressão: `flag_apto_treino = 1` + `sample_weight_contrato`;
- classificação: `flag_apto_classificacao = 1` + `sample_weight_classificacao`;
- usa `split_modelagem` pronto do notebook 01;
- seleciona apenas colunas marcadas como `feature_candidata`;
- bloqueia colunas de controle, targets e datas finais;
- validação escolhe modelo e threshold; teste fica reservado para avaliação final.


In [1]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, mean_absolute_error,
    mean_squared_error, median_absolute_error, precision_score, r2_score,
    recall_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
RANDOM_STATE = 42


## 1. Localização dos arquivos

In [2]:

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p.resolve()
    return None

DATASET_PATH = first_existing([
    PROJECT_ROOT / "data/processed/dataset_modelagem_obras_snapshots_enriquecido.csv",
    PROJECT_ROOT / "modelagem_outputs/processed/dataset_modelagem_obras_snapshots_enriquecido.csv",
    PROJECT_ROOT / "dataset_modelagem_obras_snapshots_enriquecido.csv",
    Path("/mnt/data/modelagem_outputs/processed/dataset_modelagem_obras_snapshots_enriquecido.csv"),
    Path("/mnt/data/data/processed/dataset_modelagem_obras_snapshots_enriquecido.csv"),
])

DICTIONARY_PATH = first_existing([
    PROJECT_ROOT / "data/processed/dicionario_dataset_modelagem_enriquecido.csv",
    PROJECT_ROOT / "modelagem_outputs/processed/dicionario_dataset_modelagem_enriquecido.csv",
    PROJECT_ROOT / "dicionario_dataset_modelagem_enriquecido.csv",
    Path("/mnt/data/modelagem_outputs/processed/dicionario_dataset_modelagem_enriquecido.csv"),
    Path("/mnt/data/data/processed/dicionario_dataset_modelagem_enriquecido.csv"),
])

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Dataset enriquecido não encontrado. "
        "Execute primeiro o notebook 01 enriquecido."
    )

if DICTIONARY_PATH is None:
    raise FileNotFoundError(
        "Dicionário do dataset enriquecido não encontrado."
    )

# Saída simples e independente da estrutura do projeto.
REPORTS_DIR = Path("/mnt/data/03_modelos_enriquecido_outputs")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_PATH)
print("Dicionário:", DICTIONARY_PATH)
print("Relatórios:", REPORTS_DIR.resolve())


Dataset: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dataset_modelagem_obras_snapshots_enriquecido.csv
Dicionário: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dicionario_dataset_modelagem_enriquecido.csv
Relatórios: C:\mnt\data\03_modelos_enriquecido_outputs


## 2. Carga e validação do schema

In [3]:

df = pd.read_csv(DATASET_PATH, low_memory=False, encoding="utf-8-sig")
dictionary = pd.read_csv(DICTIONARY_PATH, low_memory=False, encoding="utf-8-sig")

date_columns = [
    "data_corte", "data_os", "data_termino_original",
    "data_termino_vigente_corte", "data_conclusao_sistema_controle",
    "data_conclusao_definitiva", "data_conclusao_controle",
    "data_termino_vigente_final_controle",
]

for c in date_columns:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

required = [
    "id_contrato", "data_corte", "target_dias_atraso", "target_atraso",
    "flag_apto_treino", "flag_apto_classificacao",
    "sample_weight_contrato", "sample_weight_classificacao",
    "split_contrato_temporal", "split_modelagem",
    "dias_restantes_prazo_vigente", "data_conclusao_definitiva",
    "data_termino_vigente_final_controle",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Dataset enriquecido incompleto. Ausentes: {missing}")

if dictionary["coluna"].duplicated().any():
    raise ValueError("Dicionário possui colunas duplicadas.")

print("Shape:", df.shape)
print("Contratos:", df["id_contrato"].nunique())
print("Snapshots aptos regressão:", int(df["flag_apto_treino"].sum()))
print("Snapshots aptos classificação:", int(df["flag_apto_classificacao"].sum()))

display(
    df.groupby("split_modelagem", dropna=False)
      .agg(snapshots=("id_contrato","size"),
           contratos=("id_contrato","nunique"))
)


Shape: (633, 109)
Contratos: 48
Snapshots aptos regressão: 585
Snapshots aptos classificação: 509


,snapshots,contratos
split_modelagem,,
nao_supervisionado,48,48
test,149,8
train,372,33
validation,64,7


## 3. Validações metodológicas

In [4]:

reg_data = df[
    df["flag_apto_treino"].eq(1)
    & df["split_modelagem"].isin(["train","validation","test"])
].copy()

clf_data = df[
    df["flag_apto_classificacao"].eq(1)
    & df["split_modelagem"].isin(["train","validation","test"])
].copy()

assert not reg_data.empty
assert not clf_data.empty
assert reg_data.groupby("id_contrato")["split_modelagem"].nunique().max() == 1
assert clf_data.groupby("id_contrato")["split_modelagem"].nunique().max() == 1
assert (reg_data["data_corte"] < reg_data["data_conclusao_definitiva"]).all()
assert clf_data["dias_restantes_prazo_vigente"].ge(0).all()

target_recalc = (
    reg_data["data_conclusao_definitiva"]
    - reg_data["data_termino_vigente_final_controle"]
).dt.days.clip(lower=0)

assert np.allclose(target_recalc.astype(float), reg_data["target_dias_atraso"].astype(float))
assert np.array_equal(
    (reg_data["target_dias_atraso"] > 0).astype(int).to_numpy(),
    reg_data["target_atraso"].astype(int).to_numpy()
)

reg_weight_totals = reg_data.groupby("id_contrato")["sample_weight_contrato"].sum()
clf_weight_totals = clf_data.groupby("id_contrato")["sample_weight_classificacao"].sum()

assert np.allclose(reg_weight_totals.values, reg_weight_totals.iloc[0])
assert np.allclose(clf_weight_totals.values, clf_weight_totals.iloc[0])

contract_targets = (
    reg_data.sort_values("data_corte")
            .drop_duplicates("id_contrato")
            [["id_contrato","split_modelagem","target_atraso","target_dias_atraso"]]
)

display(
    contract_targets.groupby("split_modelagem").agg(
        contratos=("id_contrato","nunique"),
        atrasados=("target_atraso","sum"),
        taxa_atraso=("target_atraso","mean"),
        media_atraso_dias=("target_dias_atraso","mean"),
        mediana_atraso_dias=("target_dias_atraso","median"),
    )
)


,contratos,atrasados,taxa_atraso,media_atraso_dias,mediana_atraso_dias
split_modelagem,,,,,
test,8,2,0.2500,21.8750,0.0000
train,33,16,0.4848,39.0303,0.0000
validation,7,2,0.2857,12.7143,0.0000


## 4. Seleção automática das features

In [5]:

BLOCKED_COLUMNS = {
    "id_contrato","data_corte","data_os","data_termino_original",
    "data_termino_vigente_corte","status_final_controle",
    "data_conclusao_sistema_controle","data_conclusao_definitiva",
    "data_conclusao_controle","data_termino_vigente_final_controle",
    "diferenca_vigente_original_final_controle",
    "target_dias_atraso","target_atraso",
    "flag_apto_treino","flag_apto_classificacao",
    "sample_weight_contrato","sample_weight_classificacao",
    "split_contrato_temporal","split_modelagem",
}

feature_candidates = (
    dictionary.loc[dictionary["papel"].eq("feature_candidata"), "coluna"]
              .astype(str).tolist()
)

features = [
    c for c in feature_candidates
    if c in df.columns and c not in BLOCKED_COLUMNS
]

explicit_categorical = {
    "classificacao_contrato",
    "id_setor_divisao",
    "faixa_antecedencia",
}

categorical_features = [
    c for c in features
    if (
        c in explicit_categorical
        or pd.api.types.is_object_dtype(df[c])
        or isinstance(df[c].dtype, pd.CategoricalDtype)
    )
]

numeric_features = [c for c in features if c not in categorical_features]

train_reg = reg_data[reg_data["split_modelagem"].eq("train")]

def usable(cols):
    out = []
    for c in cols:
        s = train_reg[c].dropna()
        if len(s) == 0:
            continue
        if s.nunique() <= 1:
            continue
        out.append(c)
    return out

numeric_features = usable(numeric_features)
categorical_features = usable(categorical_features)
features = numeric_features + categorical_features

if not features:
    raise ValueError("Nenhuma feature disponível.")

assert not (set(features) & BLOCKED_COLUMNS)

print("Candidatas no dicionário:", len(feature_candidates))
print("Usadas:", len(features))
print("Numéricas:", len(numeric_features))
print("Categóricas:", len(categorical_features))
print("Categóricas usadas:", categorical_features)


Candidatas no dicionário: 90
Usadas: 90
Numéricas: 87
Categóricas: 3
Categóricas usadas: ['classificacao_contrato', 'id_setor_divisao', 'faixa_antecedencia']


## 5. Preprocessamento

In [6]:

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessing = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

def split_regression(data, split):
    part = data[data["split_modelagem"].eq(split)].copy()
    return (
        part,
        part[features],
        part["target_dias_atraso"].astype(float),
        part["sample_weight_contrato"].astype(float),
    )

def split_classification(data, split):
    part = data[data["split_modelagem"].eq(split)].copy()
    return (
        part,
        part[features],
        part["target_atraso"].astype(int),
        part["sample_weight_classificacao"].astype(float),
    )


# Parte A — Regressão

In [7]:

reg_train, Xr_train, yr_train, wr_train = split_regression(reg_data, "train")
reg_val, Xr_val, yr_val, wr_val = split_regression(reg_data, "validation")
reg_test, Xr_test, yr_test, wr_test = split_regression(reg_data, "test")

regression_models = {
    "dummy_mediana": Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", DummyRegressor(strategy="median")),
    ]),
    "ridge_alpha_1": Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", Ridge(alpha=1.0)),
    ]),
    "ridge_alpha_10": Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", Ridge(alpha=10.0)),
    ]),
    "ridge_alpha_100": Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", Ridge(alpha=100.0)),
    ]),
}

def regression_metrics(y_true, y_pred, sample_weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(0, np.asarray(y_pred, dtype=float))
    error = y_pred - y_true
    return {
        "mae_ponderado": mean_absolute_error(y_true, y_pred, sample_weight=sample_weight),
        "medae": median_absolute_error(y_true, y_pred),
        "rmse_ponderado": np.sqrt(mean_squared_error(y_true, y_pred, sample_weight=sample_weight)),
        "r2_ponderado": r2_score(y_true, y_pred, sample_weight=sample_weight),
        "pct_erro_abs_ate_30d": np.average(np.abs(error) <= 30, weights=sample_weight),
        "erro_assinado_medio": np.average(error, weights=sample_weight),
    }

reg_models_fitted = {}
reg_results = []

for name, model in regression_models.items():
    model.fit(Xr_train, yr_train, model__sample_weight=wr_train)
    reg_models_fitted[name] = model

    for split_name, X, y, w in [
        ("validation", Xr_val, yr_val, wr_val),
        ("test", Xr_test, yr_test, wr_test),
    ]:
        pred = np.maximum(0, model.predict(X))
        reg_results.append({
            "modelo": name,
            "split": split_name,
            **regression_metrics(y, pred, w),
        })

reg_results = pd.DataFrame(reg_results)

display(reg_results.sort_values(["split","mae_ponderado"]))

BEST_REGRESSION_MODEL = (
    reg_results[reg_results["split"].eq("validation")]
    .sort_values(["mae_ponderado","rmse_ponderado"])
    .iloc[0]["modelo"]
)

print("Melhor regressão pela validação:", BEST_REGRESSION_MODEL)


,modelo,split,mae_ponderado,medae,rmse_ponderado,r2_ponderado,pct_erro_abs_ate_30d,erro_assinado_medio
1,dummy_mediana,test,21.8750,0.0000,54.3519,-0.1933,0.8750,-21.8750
7,ridge_alpha_100,test,107.4480,11.6111,204.9190,-15.9622,0.5337,69.5762
3,ridge_alpha_1,test,142.0369,2.2127,264.8445,-27.3335,0.3797,104.0369
5,ridge_alpha_10,test,146.1165,0.0000,299.0966,-35.1361,0.4737,108.1165
0,dummy_mediana,validation,12.7143,14.0000,28.8370,-0.2413,0.8571,-12.7143
6,ridge_alpha_100,validation,259.2515,75.0000,415.7334,-256.9930,0.4048,237.8229
2,ridge_alpha_1,validation,340.4516,72.5024,553.9255,-457.0162,0.2835,320.3752
4,ridge_alpha_10,validation,376.7754,75.0000,616.4525,-566.2538,0.3095,355.3469


Melhor regressão pela validação: dummy_mediana


## 6. Regressão por horizonte

In [8]:

best_reg = reg_models_fitted[BEST_REGRESSION_MODEL]

reg_test_eval = reg_test.copy()
reg_test_eval["pred_dias_atraso"] = np.maximum(0, best_reg.predict(Xr_test))

def nearest_horizon(data, horizon):
    subset = data[data["dias_restantes_prazo_vigente"].ge(0)].copy()
    subset["dist_horizonte"] = (
        subset["dias_restantes_prazo_vigente"] - horizon
    ).abs()
    return (
        subset.sort_values(["id_contrato","dist_horizonte","data_corte"])
              .groupby("id_contrato", as_index=False)
              .first()
    )

rows = []
for horizon in [180,90,60,30]:
    h = nearest_horizon(reg_test_eval, horizon)
    rows.append({
        "horizonte_dias": horizon,
        "contratos": h["id_contrato"].nunique(),
        "mae_contrato": mean_absolute_error(h["target_dias_atraso"], h["pred_dias_atraso"]),
        "rmse_contrato": np.sqrt(mean_squared_error(h["target_dias_atraso"], h["pred_dias_atraso"])),
    })

reg_horizon_results = pd.DataFrame(rows)
display(reg_horizon_results)


,horizonte_dias,contratos,mae_contrato,rmse_contrato
0,180,8,21.8750,54.3519
1,90,8,21.8750,54.3519
2,60,8,21.8750,54.3519
3,30,8,21.8750,54.3519


# Parte B — Classificação

In [9]:

clf_train, Xc_train, yc_train, wc_train = split_classification(clf_data, "train")
clf_val, Xc_val, yc_val, wc_val = split_classification(clf_data, "validation")
clf_test, Xc_test, yc_test, wc_test = split_classification(clf_data, "test")

for name, part, y in [
    ("train", clf_train, yc_train),
    ("validation", clf_val, yc_val),
    ("test", clf_test, yc_test),
]:
    print(
        name,
        "| snapshots:", len(part),
        "| contratos:", part["id_contrato"].nunique(),
        "| taxa atraso:", f"{y.mean():.1%}",
        "| classes:", sorted(y.unique().tolist()),
    )

if yc_train.nunique() < 2:
    raise ValueError("Treino de classificação possui somente uma classe.")

classification_models = {
    "dummy_prior": Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", DummyClassifier(strategy="prior", random_state=RANDOM_STATE)),
    ]),
    "logistic_C_0_1": Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", LogisticRegression(
            C=0.1, max_iter=5000, solver="liblinear",
            random_state=RANDOM_STATE
        )),
    ]),
    "logistic_C_1": Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", LogisticRegression(
            C=1.0, max_iter=5000, solver="liblinear",
            random_state=RANDOM_STATE
        )),
    ]),
    "logistic_balanced_C_0_1": Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", LogisticRegression(
            C=0.1, max_iter=5000, solver="liblinear",
            class_weight="balanced", random_state=RANDOM_STATE
        )),
    ]),
    "logistic_balanced_C_1": Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", LogisticRegression(
            C=1.0, max_iter=5000, solver="liblinear",
            class_weight="balanced", random_state=RANDOM_STATE
        )),
    ]),
}

def safe_auc(y_true, prob, sample_weight=None):
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, prob, sample_weight=sample_weight)

def safe_ap(y_true, prob, sample_weight=None):
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return average_precision_score(y_true, prob, sample_weight=sample_weight)

def classification_metrics(y_true, prob, threshold, sample_weight):
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy_ponderada": accuracy_score(y_true, pred, sample_weight=sample_weight),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred, sample_weight=sample_weight),
        "precision": precision_score(y_true, pred, sample_weight=sample_weight, zero_division=0),
        "recall": recall_score(y_true, pred, sample_weight=sample_weight, zero_division=0),
        "f1": f1_score(y_true, pred, sample_weight=sample_weight, zero_division=0),
        "brier": brier_score_loss(y_true, prob, sample_weight=sample_weight),
        "roc_auc": safe_auc(y_true, prob, sample_weight),
        "pr_auc": safe_ap(y_true, prob, sample_weight),
    }

clf_models_fitted = {}
clf_results_05 = []

for name, model in classification_models.items():
    model.fit(Xc_train, yc_train, model__sample_weight=wc_train)
    clf_models_fitted[name] = model

    for split_name, X, y, w in [
        ("validation", Xc_val, yc_val, wc_val),
        ("test", Xc_test, yc_test, wc_test),
    ]:
        prob = model.predict_proba(X)[:,1]
        clf_results_05.append({
            "modelo": name,
            "split": split_name,
            **classification_metrics(y, prob, 0.50, w),
        })

clf_results_05 = pd.DataFrame(clf_results_05)
display(
    clf_results_05.sort_values(
        ["split","roc_auc"],
        ascending=[True,False]
    )
)


train | snapshots: 311 | contratos: 33 | taxa atraso: 44.1% | classes: [0, 1]
validation | snapshots: 61 | contratos: 7 | taxa atraso: 50.8% | classes: [0, 1]
test | snapshots: 137 | contratos: 8 | taxa atraso: 11.7% | classes: [0, 1]


,modelo,split,threshold,accuracy_ponderada,balanced_accuracy,precision,recall,f1,brier,roc_auc,pr_auc
1,dummy_prior,test,0.5000,0.7500,0.5000,0.0000,0.0000,0.0000,0.2427,0.5000,0.2500
5,logistic_C_1,test,0.5000,0.3833,0.2556,0.0000,0.0000,0.0000,0.5903,0.0113,0.1493
9,logistic_balanced_C_1,test,0.5000,0.3833,0.2556,0.0000,0.0000,0.0000,0.5908,0.0113,0.1493
3,logistic_C_0_1,test,0.5000,0.3833,0.2556,0.0000,0.0000,0.0000,0.5618,0.0090,0.1491
7,logistic_balanced_C_0_1,test,0.5000,0.3833,0.2556,0.0000,0.0000,0.0000,0.5631,0.0090,0.1491
0,dummy_prior,validation,0.5000,0.7143,0.5000,0.0000,0.0000,0.0000,0.2437,0.5000,0.2857
4,logistic_C_1,validation,0.5000,0.3377,0.2364,0.0000,0.0000,0.0000,0.6048,0.4111,0.2441
8,logistic_balanced_C_1,validation,0.5000,0.3377,0.2364,0.0000,0.0000,0.0000,0.6049,0.4111,0.2441
2,logistic_C_0_1,validation,0.5000,0.3377,0.2364,0.0000,0.0000,0.0000,0.5516,0.4000,0.2421
6,logistic_balanced_C_0_1,validation,0.5000,0.3247,0.2273,0.0000,0.0000,0.0000,0.5527,0.4000,0.2421


## 7. Seleção do classificador e threshold

In [10]:

validation_clf = clf_results_05[
    clf_results_05["split"].eq("validation")
].copy()

if validation_clf["roc_auc"].notna().any():
    validation_clf = validation_clf.sort_values(
        ["roc_auc","pr_auc","balanced_accuracy"],
        ascending=[False,False,False]
    )
else:
    validation_clf = validation_clf.sort_values(
        ["balanced_accuracy","f1"],
        ascending=[False,False]
    )

BEST_CLASSIFICATION_MODEL = validation_clf.iloc[0]["modelo"]
best_clf = clf_models_fitted[BEST_CLASSIFICATION_MODEL]

prob_val = best_clf.predict_proba(Xc_val)[:,1]
threshold_rows = []

for threshold in np.arange(0.05,0.951,0.01):
    threshold_rows.append(
        classification_metrics(
            yc_val, prob_val, threshold, wc_val
        )
    )

threshold_results = pd.DataFrame(threshold_rows)

BEST_THRESHOLD = float(
    threshold_results.sort_values(
        ["balanced_accuracy","f1","precision"],
        ascending=[False,False,False]
    ).iloc[0]["threshold"]
)

prob_test = best_clf.predict_proba(Xc_test)[:,1]
pred_test = (prob_test >= BEST_THRESHOLD).astype(int)

final_classification = pd.DataFrame([
    {
        "split": "validation",
        **classification_metrics(
            yc_val, prob_val, BEST_THRESHOLD, wc_val
        )
    },
    {
        "split": "test",
        **classification_metrics(
            yc_test, prob_test, BEST_THRESHOLD, wc_test
        )
    },
])

print("Melhor classificador:", BEST_CLASSIFICATION_MODEL)
print("Threshold:", round(BEST_THRESHOLD,2))
display(final_classification)

print("Matriz de confusão — teste:")
print(confusion_matrix(yc_test, pred_test))


Melhor classificador: dummy_prior
Threshold: 0.05


,split,threshold,accuracy_ponderada,balanced_accuracy,precision,recall,f1,brier,roc_auc,pr_auc
0,validation,0.0500,0.2857,0.5000,0.2857,1.0000,0.4444,0.2437,0.5000,0.2857
1,test,0.0500,0.2500,0.5000,0.2500,1.0000,0.4000,0.2427,0.5000,0.2500


Matriz de confusão — teste:
[[  0 121]
 [  0  16]]


## 8. Avaliação por contrato em horizontes

In [11]:

clf_test_eval = clf_test.copy()
clf_test_eval["prob_atraso"] = prob_test
clf_test_eval["pred_atraso"] = pred_test

def horizon_classification_metrics(data, horizon):
    subset = data.copy()
    subset["dist_horizonte"] = (
        subset["dias_restantes_prazo_vigente"] - horizon
    ).abs()

    subset = (
        subset.sort_values(["id_contrato","dist_horizonte","data_corte"])
              .groupby("id_contrato", as_index=False)
              .first()
    )

    if subset["target_atraso"].nunique() > 1:
        bac = balanced_accuracy_score(subset["target_atraso"], subset["pred_atraso"])
        auc = roc_auc_score(subset["target_atraso"], subset["prob_atraso"])
        ap = average_precision_score(subset["target_atraso"], subset["prob_atraso"])
    else:
        bac = auc = ap = np.nan

    return {
        "horizonte_dias": horizon,
        "contratos": subset["id_contrato"].nunique(),
        "taxa_atraso": subset["target_atraso"].mean(),
        "balanced_accuracy": bac,
        "roc_auc": auc,
        "pr_auc": ap,
    }, subset

rows = []
detail_90 = None

for horizon in [180,90,60,30]:
    result, detail = horizon_classification_metrics(clf_test_eval, horizon)
    rows.append(result)
    if horizon == 90:
        detail_90 = detail

clf_horizon_results = pd.DataFrame(rows)
display(clf_horizon_results)

print("Detalhe ~90 dias:")
display(
    detail_90[
        [
            "id_contrato","data_corte","dias_restantes_prazo_vigente",
            "target_atraso","prob_atraso","pred_atraso"
        ]
    ]
)


,horizonte_dias,contratos,taxa_atraso,balanced_accuracy,roc_auc,pr_auc
0,180,8,0.2500,0.5000,0.5000,0.2500
1,90,8,0.2500,0.5000,0.5000,0.2500
2,60,8,0.2500,0.5000,0.5000,0.2500
3,30,8,0.2500,0.5000,0.5000,0.2500


Detalhe ~90 dias:


,id_contrato,data_corte,dias_restantes_prazo_vigente,target_atraso,prob_atraso,pred_atraso
0,173,2024-07-31,80,0,0.4848,1
1,201,2024-11-30,102,0,0.4848,1
2,254,2025-05-31,83,0,0.4848,1
3,283,2024-11-30,97,0,0.4848,1
4,287,2025-03-31,84,0,0.4848,1
5,317,2024-11-30,84,1,0.4848,1
6,342,2025-07-18,60,0,0.4848,1
7,377,2025-04-29,90,1,0.4848,1


## 9. Coeficientes dos melhores modelos lineares

In [12]:

def get_feature_names(fitted_pipeline, sample_X):
    pre = fitted_pipeline["preprocessing"]
    try:
        return pre.get_feature_names_out()
    except Exception:
        n = pre.transform(sample_X.iloc[:1]).shape[1]
        return np.array([f"feature_{i}" for i in range(n)])

reg_importance = pd.DataFrame()
if BEST_REGRESSION_MODEL != "dummy_mediana":
    model = reg_models_fitted[BEST_REGRESSION_MODEL]
    names = get_feature_names(model, Xr_train)
    coef = np.asarray(model["model"].coef_)
    reg_importance = pd.DataFrame({
        "feature": names,
        "coeficiente": coef,
    })
    reg_importance["abs_coeficiente"] = reg_importance["coeficiente"].abs()
    display(
        reg_importance.sort_values(
            "abs_coeficiente",
            ascending=False
        ).head(20)
    )

clf_importance = pd.DataFrame()
if BEST_CLASSIFICATION_MODEL != "dummy_prior":
    model = clf_models_fitted[BEST_CLASSIFICATION_MODEL]
    names = get_feature_names(model, Xc_train)
    coef = np.asarray(model["model"].coef_[0])
    clf_importance = pd.DataFrame({
        "feature": names,
        "coeficiente": coef,
    })
    clf_importance["abs_coeficiente"] = clf_importance["coeficiente"].abs()
    display(
        clf_importance.sort_values(
            "abs_coeficiente",
            ascending=False
        ).head(20)
    )


## 10. Sanity checks e exportação

In [13]:

assert not (set(features) & BLOCKED_COLUMNS)
assert clf_data["dias_restantes_prazo_vigente"].ge(0).all()
assert reg_data.groupby("id_contrato")["split_modelagem"].nunique().max() == 1
assert clf_data.groupby("id_contrato")["split_modelagem"].nunique().max() == 1

REG_METRICS_PATH = REPORTS_DIR / "03_enriquecido_regressao.csv"
REG_HORIZON_PATH = REPORTS_DIR / "03_enriquecido_regressao_horizontes.csv"
CLF_05_PATH = REPORTS_DIR / "03_enriquecido_classificacao_threshold_050.csv"
CLF_FINAL_PATH = REPORTS_DIR / "03_enriquecido_classificacao_final.csv"
CLF_HORIZON_PATH = REPORTS_DIR / "03_enriquecido_classificacao_horizontes.csv"
THRESHOLD_PATH = REPORTS_DIR / "03_enriquecido_thresholds_validacao.csv"
FEATURES_PATH = REPORTS_DIR / "03_enriquecido_features_utilizadas.csv"

reg_results.to_csv(REG_METRICS_PATH, index=False, encoding="utf-8-sig")
reg_horizon_results.to_csv(REG_HORIZON_PATH, index=False, encoding="utf-8-sig")
clf_results_05.to_csv(CLF_05_PATH, index=False, encoding="utf-8-sig")
final_classification.to_csv(CLF_FINAL_PATH, index=False, encoding="utf-8-sig")
clf_horizon_results.to_csv(CLF_HORIZON_PATH, index=False, encoding="utf-8-sig")
threshold_results.to_csv(THRESHOLD_PATH, index=False, encoding="utf-8-sig")

pd.DataFrame({
    "feature": features,
    "tipo": [
        "categorica" if f in categorical_features else "numerica"
        for f in features
    ]
}).to_csv(FEATURES_PATH, index=False, encoding="utf-8-sig")

if len(reg_importance):
    reg_importance.to_csv(
        REPORTS_DIR / "03_enriquecido_coeficientes_regressao.csv",
        index=False, encoding="utf-8-sig"
    )

if len(clf_importance):
    clf_importance.to_csv(
        REPORTS_DIR / "03_enriquecido_coeficientes_classificacao.csv",
        index=False, encoding="utf-8-sig"
    )

print("=" * 72)
print("NOTEBOOK 03 — DATASET ENRIQUECIDO")
print("=" * 72)
print("Features utilizadas:", len(features))
print("Melhor regressão:", BEST_REGRESSION_MODEL)
print("Melhor classificação:", BEST_CLASSIFICATION_MODEL)
print("Threshold:", round(BEST_THRESHOLD,2))
print("Relatórios:", REPORTS_DIR.resolve())
print("Sanity checks concluídos com sucesso.")


NOTEBOOK 03 — DATASET ENRIQUECIDO
Features utilizadas: 90
Melhor regressão: dummy_mediana
Melhor classificação: dummy_prior
Threshold: 0.05
Relatórios: C:\mnt\data\03_modelos_enriquecido_outputs
Sanity checks concluídos com sucesso.
